# Expresiones verbales de probabilidad — línea de base humana vs. muestra de silicio

Segunda parte del taller. En el [ejercicio 1](https://github.com/gefero/factor_data_silicon_tutorial)
simulamos respuestas a la WVS y medimos el sesgo de punto medio de forma **indirecta**: la escala de
Q130 tiene 4 categorías y ningún centro, así que el sesgo había que leerlo como concentración en las
categorías interiores.

Acá la escala es **continua de 0 a 100 y sí tiene punto medio**, así que el sesgo se puede medir
directamente. Y la línea de base no es una encuesta ya publicada: son las respuestas que dio
**este curso** en el formulario.

**Diseño experimental**

- Variables de perfil: edad, género, nivel educativo, país (lo que releva el formulario)
- **26 llamadas independientes por participante** — 16 del bloque puntual + 10 del bloque de rango
- Cada llamada recibe solo el perfil y **una** expresión, sin contexto de las otras
- 2 versiones de prompt: `v1` neutral (línea de base) y `v2` con instrucción anti-punto-medio

| Bloque | Ítems | Escala | Llamadas |
|---|---|---|---|
| Valor puntual | 16 expresiones | 0–100 | 16 |
| Rango | 10 de esas 16 | mín–máx 0–100 | 10 |

**Por qué 26 llamadas y no una sola.** Sería mucho más barato pedirle al modelo las 16 expresiones
en un solo JSON. Pero ahí el modelo las ve todas juntas y las ordena de forma coherente entre sí,
una ventaja que las personas no tuvieron: cada participante vio una expresión por vez, en orden
aleatorio, sin poder volver atrás a comparar. Si queremos comparar dispersión y coherencia interna
contra humanos, las condiciones tienen que ser equivalentes.

**Por qué dos versiones de prompt.** Si el prompt dice "no uses 50", ya mitigamos el sesgo antes de
medirlo. `v1` es la línea de base y `v2` la mitigación, igual que los prompts 1 y 2 del ejercicio 1.
Corran las dos y comparen.

**Backends**: OpenAI API · Ollama (local)

## 0. Dependencias

In [ ]:
# PARA USAR OPENAI
#!pip install openai pandas matplotlib seaborn tqdm

# PARA USAR OLLAMA (modelos de pesos abiertos en el entorno de Colab)
#!pip install ollama pandas matplotlib seaborn tqdm

## 1. Configuración

In [ ]:
import json
import os
import re
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Parámetros principales ──────────────────────────────────────────────────
BACKEND     = "openai"          # "openai" | "ollama"
MODEL       = "gpt-4o-mini"     # openai: "gpt-4o", "gpt-4o-mini", "gpt-4.1"
                                # ollama: "gpt-oss:20b", "llama3.1", etc.
TEMPERATURE = 1.0

# Versión del prompt:
#   "v1" → neutral, sin instrucciones sobre el punto medio  (LÍNEA DE BASE)
#   "v2" → agrega la instrucción anti-punto-medio           (MITIGACIÓN)
# Corran las dos y comparen. Con v2 solo, no hay sesgo que medir.
VERSION_PROMPT = "v1"

# ── Hoja con las respuestas humanas ─────────────────────────────────────────
# La hoja tiene que estar compartida: Compartir → Acceso general →
# "Cualquier persona con el enlace" → Lector.
SHEET_ID = "1V_3baEk7XSe36Mdlvn47bVgDBBcbMaYE8NmguHS9Zn8"
HOJA     = "respuestas"          # nombre de la pestaña, no el gid

# N_SAMPLE: participantes humanos a simular.
#   None → todos
#   int  → muestra aleatoria de ese tamaño
# Ojo con el costo: son 26 llamadas por participante.
#   n=30 → 780 llamadas.
N_SAMPLE    = None
RANDOM_SEED = 42

# Pausa entre llamadas (segundos). OpenAI: 0.5 para evitar rate-limit | Ollama: 0
SLEEP = 0.5

OUTPUT_CSV = f"/content/{MODEL.replace(':', '-')}_{VERSION_PROMPT}_expresiones_silicon.csv"

sns.set_style("whitegrid")

print(f"Backend  : {BACKEND}")
print(f"Modelo   : {MODEL}")
print(f"Prompt   : {VERSION_PROMPT}")
print(f"Salida   : {OUTPUT_CSV}")

## 2. Carga de las respuestas humanas

El formulario escribe una fila por participante y **74 columnas**: `punto_<expr>` para las 16
expresiones, `min_<expr>` / `max_<expr>` para las 10 con rango, las latencias `*ms_<expr>`, los
demográficos y el orden efectivo en que se presentaron los ítems.

Para trabajar lo pasamos a **formato largo**: una fila por participante × expresión.

In [ ]:
# Las 16 expresiones, en el mismo orden y con las mismas claves que el formulario.
EXPRESIONES = [
    {"key": "seguro",         "palabra": "Seguro",         "rango": True},
    {"key": "casi_seguro",    "palabra": "Casi seguro",    "rango": True},
    {"key": "seguramente",    "palabra": "Seguramente",    "rango": False},
    {"key": "muy_probable",   "palabra": "Muy probable",   "rango": True},
    {"key": "pienso_que_si",  "palabra": "Pienso que sí",  "rango": False},
    {"key": "creo_que_si",    "palabra": "Creo que sí",    "rango": True},
    {"key": "probable",       "palabra": "Probable",       "rango": False},
    {"key": "puede_ser",      "palabra": "Puede ser",      "rango": True},
    {"key": "quizas",         "palabra": "Quizás",         "rango": True},
    {"key": "igual_al_azar",  "palabra": "Igual al azar",  "rango": True},
    {"key": "no_se",          "palabra": "No sé",          "rango": True},
    {"key": "lo_dudo",        "palabra": "Lo dudo",        "rango": True},
    {"key": "poco_probable",  "palabra": "Poco probable",  "rango": False},
    {"key": "dificilmente",   "palabra": "Difícilmente",   "rango": True},
    {"key": "improbable",     "palabra": "Improbable",     "rango": False},
    {"key": "muy_improbable", "palabra": "Muy improbable", "rango": False},
]
CLAVES     = [e["key"] for e in EXPRESIONES]
PALABRA    = {e["key"]: e["palabra"] for e in EXPRESIONES}
CON_RANGO  = [e["key"] for e in EXPRESIONES if e["rango"]]

URL_HOJA = (f"https://docs.google.com/spreadsheets/d/{SHEET_ID}"
            f"/gviz/tq?tqx=out:csv&sheet={HOJA}")

humanos_ancho = pd.read_csv(URL_HOJA)
print(f"Respuestas humanas crudas : {len(humanos_ancho)}")

# Descartamos filas incompletas (alguien que cerró la pestaña a mitad de camino).
cols_punto = [f"punto_{k}" for k in CLAVES]
completas  = humanos_ancho[cols_punto].notna().all(axis=1)
if (~completas).any():
    print(f"Descartadas por incompletas: {(~completas).sum()}")
humanos_ancho = humanos_ancho[completas].copy()

if N_SAMPLE is not None and N_SAMPLE < len(humanos_ancho):
    humanos_ancho = humanos_ancho.sample(N_SAMPLE, random_state=RANDOM_SEED)
    print(f"Muestreados: {len(humanos_ancho)}")

PERFIL_COLS = ["edad", "genero", "nivel_educativo", "pais", "provincia"]


def a_largo(df_ancho: pd.DataFrame, fuente: str) -> pd.DataFrame:
    """Una fila por participante × expresión."""
    filas = []
    for _, r in df_ancho.iterrows():
        for k in CLAVES:
            filas.append({
                "participant_id": r["participant_id"],
                "fuente"        : fuente,
                "expresion"     : k,
                "palabra"       : PALABRA[k],
                "punto"         : r.get(f"punto_{k}"),
                "min"           : r.get(f"min_{k}") if k in CON_RANGO else np.nan,
                "max"           : r.get(f"max_{k}") if k in CON_RANGO else np.nan,
                **{c: r.get(c) for c in PERFIL_COLS},
            })
    return pd.DataFrame(filas)


humanos = a_largo(humanos_ancho, "humano")

print(f"\nParticipantes : {humanos['participant_id'].nunique()}")
print(f"Filas (largo) : {len(humanos)}")
print("\nPerfil de la muestra:")
for c in ["genero", "nivel_educativo", "pais"]:
    print(f"\n{c}:")
    print(humanos_ancho[c].value_counts().to_string())
print(f"\nEdad: mediana {humanos_ancho['edad'].median():.0f}, "
      f"rango {humanos_ancho['edad'].min():.0f}-{humanos_ancho['edad'].max():.0f}")

humanos.head(3)

## 3. System prompts y builders

El perfil se arma en **primera persona**, a partir de lo único que
releva el formulario: edad, género, nivel educativo y país. Es un perfil menos informativo que el ejercicio anterior y se puede discutir cuánta variabilidad puede reproducir un modelo con tan poco.

Los prompts van **en español** porque el
estímulo *son* las palabras en español. "Puede ser" y "maybe" no tienen por qué ocupar el mismo
lugar en la escala, y preguntar en inglés por una expresión castellana cambia el objeto de estudio.

In [ ]:
# ── Diccionarios: los slugs del CSV a texto legible ─────────────────────────
GENERO = {
    "mujer": "una mujer", "varon": "un varón", "no-bin": "una persona no binaria",
    "gen-fluido": "una persona de género fluido", "no-decirlo": "una persona",
}
EDUCACION = {
    "primario-incompleto": "primario incompleto", "primario-completo": "primario completo",
    "secundario-incompleto": "secundario incompleto", "secundario-completo": "secundario completo",
    "terciario-incompleto": "terciario incompleto", "terciario-completo": "terciario completo",
    "universidad-incompleta": "universitario incompleto",
    "universidad-completa": "universitario completo",
    "posgrado-incompleto": "posgrado incompleto", "posgrado-completo": "posgrado completo",
}
PAIS = {
    "argentina": "Argentina", "uruguay": "Uruguay", "chile": "Chile", "paraguay": "Paraguay",
    "bolivia": "Bolivia", "brasil": "Brasil", "peru": "Perú", "colombia": "Colombia",
    "mexico": "México", "espana": "España", "estados-unidos": "Estados Unidos",
    "alemania": "Alemania", "otro": "otro país",
}


def build_perfil(row: pd.Series) -> str:
    genero = GENERO.get(row["genero"], "una persona")
    educ   = EDUCACION.get(row["nivel_educativo"], row["nivel_educativo"])
    pais   = PAIS.get(row["pais"], row["pais"])
    lugar  = pais
    if row["pais"] == "argentina" and isinstance(row.get("provincia"), str) and row["provincia"]:
        lugar = f"{row['provincia'].replace('-', ' ').title()}, Argentina"
    return (
        f"Soy {genero} de {int(row['edad'])} años y vivo en {lugar}. "
        f"Mi máximo nivel educativo alcanzado es {educ}."
    )


# ── v1 — LÍNEA DE BASE. Sin ninguna instrucción sobre el punto medio ────────
SYSTEM_PUNTO_V1 = """Vas a recibir la autodescripción de una persona que participa en
una encuesta. Tu tarea es responder la consigna como la respondería esa persona,
adoptando su perspectiva y su forma de usar el lenguaje.

Reglas:
1. Respondé con un número entero entre 0 y 100.
2. Razoná brevemente en el campo "thinking" (1-2 oraciones), en primera persona,
   como la persona encuestada.
3. Respondé EXCLUSIVAMENTE en el formato JSON indicado. Sin texto adicional.
"""

# ── v2 — MITIGACIÓN. Igual, más las reglas anti-punto-medio ─────────────────
SYSTEM_PUNTO_V2 = """Vas a recibir la autodescripción de una persona que participa en
una encuesta. Tu tarea es responder la consigna como la respondería esa persona,
adoptando su perspectiva y su forma de usar el lenguaje.

Reglas:
1. Respondé con un número entero entre 0 y 100.
2. NO recurras por defecto a 50 ni a valores cercanos al centro de la escala.
   En esta escala 50 significa "exactamente tan probable como improbable", que es
   una afirmación fuerte y precisa — no es una forma de decir "no sé".
3. Toda la escala es válida. Los valores extremos (por debajo de 10, por encima de
   90) son respuestas legítimas y frecuentes: asignalos cuando la expresión lo
   amerite.
4. Las personas difieren mucho entre sí en cómo interpretan estas expresiones.
   Respondé como esta persona en particular, no como el promedio de la población.
5. Razoná brevemente en el campo "thinking" (1-2 oraciones), en primera persona,
   como la persona encuestada.
6. Respondé EXCLUSIVAMENTE en el formato JSON indicado. Sin texto adicional.
"""

SYSTEM_RANGO_V1 = """Vas a recibir la autodescripción de una persona que participa en
una encuesta. Tu tarea es responder la consigna como la respondería esa persona,
adoptando su perspectiva y su forma de usar el lenguaje.

Reglas:
1. Respondé con dos números enteros entre 0 y 100, con "min" menor o igual a "max".
2. Razoná brevemente en el campo "thinking" (1-2 oraciones), en primera persona,
   como la persona encuestada.
3. Respondé EXCLUSIVAMENTE en el formato JSON indicado. Sin texto adicional.
"""

SYSTEM_RANGO_V2 = """Vas a recibir la autodescripción de una persona que participa en
una encuesta. Tu tarea es responder la consigna como la respondería esa persona,
adoptando su perspectiva y su forma de usar el lenguaje.

Reglas:
1. Respondé con dos números enteros entre 0 y 100, con "min" menor o igual a "max".
2. NO centres el rango en 50 por defecto ni uses rangos simétricos por comodidad.
   Muchas expresiones corresponden a rangos angostos y desplazados hacia un extremo.
3. El ancho del rango tiene que reflejar cuánta imprecisión tiene ESA expresión en
   particular, no una incertidumbre genérica.
4. Razoná brevemente en el campo "thinking" (1-2 oraciones), en primera persona,
   como la persona encuestada.
5. Respondé EXCLUSIVAMENTE en el formato JSON indicado. Sin texto adicional.
"""


def build_prompt_punto(row: pd.Series, palabra: str) -> str:
    return (
        f"Perfil de la persona:\n{build_perfil(row)}\n\n"
        "Respondé la siguiente consigna como lo haría esta persona.\n\n"
        "Nuestro lenguaje está lleno de expresiones que usamos para comunicar distintos "
        "grados de certeza sobre una afirmación. Supongamos que una de esas expresiones "
        "describe la probabilidad de que algo pase.\n\n"
        f'¿Qué probabilidad de que el evento suceda le asignarías a la expresión "{palabra}", '
        "en una escala de 0 a 100?\n\n"
        "Devolvé SOLO este JSON (sin markdown, sin texto extra):\n"
        "{\n"
        '  "thinking": "<razonamiento breve, 1-2 oraciones>",\n'
        '  "punto": <entero entre 0 y 100>\n'
        "}"
    )


def build_prompt_rango(row: pd.Series, palabra: str) -> str:
    return (
        f"Perfil de la persona:\n{build_perfil(row)}\n\n"
        "Respondé la siguiente consigna como lo haría esta persona.\n\n"
        "En lugar de pensar en un único valor para la expresión, pensá en un rango.\n\n"
        f'¿Qué rango de probabilidad (mínimo y máximo, de 0 a 100) representa mejor a la '
        f'expresión "{palabra}"? En otras palabras, el rango de valores para el cual usarías '
        "esa expresión.\n\n"
        "Devolvé SOLO este JSON (sin markdown, sin texto extra):\n"
        "{\n"
        '  "thinking": "<razonamiento breve, 1-2 oraciones>",\n'
        '  "min": <entero entre 0 y 100>,\n'
        '  "max": <entero entre 0 y 100>\n'
        "}"
    )


SYSTEM = {
    ("punto", "v1"): SYSTEM_PUNTO_V1, ("punto", "v2"): SYSTEM_PUNTO_V2,
    ("rango", "v1"): SYSTEM_RANGO_V1, ("rango", "v2"): SYSTEM_RANGO_V2,
}

# Vista de ejemplo con el primer participante real
_ej = humanos_ancho.iloc[0]
print("=== PERFIL ===")
print(build_perfil(_ej))
print(f"\n=== SYSTEM (punto, {VERSION_PROMPT}) ===")
print(SYSTEM[("punto", VERSION_PROMPT)])
print("=== USER (punto) ===")
print(build_prompt_punto(_ej, "Quizás"))
print("\n=== USER (rango) ===")
print(build_prompt_rango(_ej, "Quizás"))

## 4. Parsing y validación

In [ ]:
def parse_response(raw: str) -> dict:
    """Mismo parser que el ejercicio 1: tolera ```json, texto alrededor, etc."""
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.IGNORECASE)
    raw = re.sub(r"\s*```$", "", raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except json.JSONDecodeError:
                pass
    return {}


def _entero_valido(v) -> bool:
    """0-100. Acepta 70 y "70", rechaza 70.5, "setenta" y None."""
    if isinstance(v, bool) or v is None:
        return False
    try:
        f = float(v)
    except (TypeError, ValueError):
        return False
    return f == int(f) and 0 <= f <= 100


def validate_punto(p: dict) -> bool:
    return _entero_valido(p.get("punto"))


def validate_rango(p: dict) -> bool:
    if not (_entero_valido(p.get("min")) and _entero_valido(p.get("max"))):
        return False
    return int(float(p["min"])) <= int(float(p["max"]))


assert validate_punto({"punto": 0}) and validate_punto({"punto": "70"})
assert not validate_punto({"punto": 101}) and not validate_punto({"punto": None})
assert not validate_punto({"punto": 70.5}) and not validate_punto({})
assert validate_rango({"min": 20, "max": 20})
assert not validate_rango({"min": 80, "max": 20})
print("OK")

## 5. Backends LLM

In [ ]:
def call_openai(system_prompt: str, user_prompt: str) -> tuple[dict, str]:
    from openai import OpenAI
    from google.colab import userdata
    api_key = userdata.get('key_openai')
    if not api_key:
        raise EnvironmentError("key_openai no encontrada en Colab secrets.")
    client = OpenAI(api_key=api_key)
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=TEMPERATURE,
    )
    raw = resp.choices[0].message.content.strip()
    return parse_response(raw), raw


def call_ollama(system_prompt: str, user_prompt: str) -> tuple[dict, str]:
    import ollama
    resp = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        options={"temperature": TEMPERATURE},
    )
    raw = resp["message"]["content"].strip()
    return parse_response(raw), raw


CALL = call_openai if BACKEND == "openai" else call_ollama


def generate_one(row: pd.Series, bloque: str, palabra: str) -> dict:
    """Una llamada: un perfil, una expresión, un bloque. Sin contexto de las otras."""
    sys_p = SYSTEM[(bloque, VERSION_PROMPT)]
    usr_p = (build_prompt_punto if bloque == "punto" else build_prompt_rango)(row, palabra)
    try:
        parsed, raw = CALL(sys_p, usr_p)
        valid = (validate_punto if bloque == "punto" else validate_rango)(parsed)
    except Exception as e:
        parsed, raw, valid = {}, str(e), False
    return {
        "punto"   : int(float(parsed["punto"])) if valid and bloque == "punto" else np.nan,
        "min"     : int(float(parsed["min"]))   if valid and bloque == "rango" else np.nan,
        "max"     : int(float(parsed["max"]))   if valid and bloque == "rango" else np.nan,
        "thinking": parsed.get("thinking", ""),
        "valid"   : valid,
        "raw"     : raw,
    }


print(f"Backend activo : {BACKEND} → {CALL.__name__}")
print(f"Prompt activo  : {VERSION_PROMPT}")
n_part = humanos_ancho.shape[0]
print(f"Llamadas totales: {n_part} participantes × 26 = {n_part * 26}")

## 6. Simulación

Cada participante genera 26 llamadas: 16 del bloque puntual y 10 del de rango.

> Guardado incremental cada 20 llamadas. Si el kernel se corta, volvé a correr esta celda:
> retoma donde quedó, sin repetir ni duplicar llamadas ya hechas.

In [ ]:
from tqdm.notebook import tqdm

out_path = Path(OUTPUT_CSV)

# Una "tarea" es (participante, bloque, expresión). La clave permite reanudar sin duplicar.
tareas = []
for _, r in humanos_ancho.iterrows():
    for k in CLAVES:
        tareas.append((r, "punto", k))
    for k in CON_RANGO:
        tareas.append((r, "rango", k))

if out_path.exists():
    df_prev = pd.read_csv(out_path)
    hechas  = set(zip(df_prev["participant_id"], df_prev["bloque"], df_prev["expresion"]))
    records = df_prev.to_dict("records")
    tareas  = [t for t in tareas if (t[0]["participant_id"], t[1], t[2]) not in hechas]
    print(f"Reanudando: {len(hechas)} llamadas hechas, {len(tareas)} pendientes.")
else:
    records = []
    print(f"Inicio fresco: {len(tareas)} llamadas — prompt {VERSION_PROMPT}.")

for i, (row, bloque, k) in enumerate(tqdm(tareas, total=len(tareas)), start=1):
    res = generate_one(row, bloque, PALABRA[k])

    records.append({
        # Identificadores
        "participant_id" : row["participant_id"],
        "fuente"         : MODEL,
        "bloque"         : bloque,
        "expresion"      : k,
        "palabra"        : PALABRA[k],
        # Perfil enviado al modelo
        "edad"           : row["edad"],
        "genero"         : row["genero"],
        "nivel_educativo": row["nivel_educativo"],
        "pais"           : row["pais"],
        "provincia"      : row.get("provincia", ""),
        # Respuesta del modelo
        "punto"          : res["punto"],
        "min"            : res["min"],
        "max"            : res["max"],
        "thinking"       : res["thinking"],
        # Ground truth humano de esa misma persona y esa misma expresión
        "punto_humano"   : row.get(f"punto_{k}"),
        "min_humano"     : row.get(f"min_{k}") if k in CON_RANGO else np.nan,
        "max_humano"     : row.get(f"max_{k}") if k in CON_RANGO else np.nan,
        # Validez y metadatos
        "valid"          : res["valid"],
        "raw_response"   : res["raw"],
        "backend"        : BACKEND,
        "model_name"     : MODEL,
        "version_prompt" : VERSION_PROMPT,
        "temperature"    : TEMPERATURE,
        "timestamp"      : datetime.utcnow().isoformat(),
    })

    if i % 20 == 0:
        pd.DataFrame(records).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    if SLEEP:
        time.sleep(SLEEP)

simulados = pd.DataFrame(records)
simulados.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\n✅ {len(simulados)} llamadas → '{OUTPUT_CSV}'")

## 7. Vista previa y calidad

In [ ]:
# Para recargar sin re-ejecutar la simulación:
# simulados = pd.read_csv(OUTPUT_CSV)

print(f"Modelo            : {simulados['model_name'].iloc[0]}")
print(f"Versión de prompt : {simulados['version_prompt'].iloc[0]}")
print(f"Llamadas totales  : {len(simulados)}")
print(f"Respuestas válidas: {simulados['valid'].mean() * 100:.1f}%")

invalidas = simulados[~simulados["valid"]]
if len(invalidas):
    print(f"\n⚠ {len(invalidas)} llamadas inválidas. Primeras respuestas crudas:")
    for r in invalidas["raw_response"].head(3):
        print(" ", str(r)[:160].replace("\n", " "))

simulados[simulados["valid"] & (simulados["bloque"] == "punto")][
    ["palabra", "edad", "genero", "punto_humano", "punto", "thinking"]
].head(6)

## 8. Métricas de sesgo de punto medio

Las cinco métricas, sobre el bloque puntual salvo la última:

1. **Masa en el punto medio** — proporción de respuestas en `[45, 55]`. En humanos debería ser alta
   solo para *Igual al azar* y *No sé*. Si el modelo la infla para *Casi seguro* o *Lo dudo*, eso es
   el sesgo.
2. **Exceso exacto en 50** — proporción de respuestas *exactamente* 50. En el instrumento humano ese
   valor solo se alcanza moviendo el slider a propósito, porque el círculo arranca oculto. En los
   LLMs el 50 es un atractor fuerte. Es probablemente la métrica más limpia del ejercicio.
3. **Dispersión entre participantes** — desvío estándar por expresión. El hallazgo central del
   estudio original es la *enorme* dispersión entre personas ("el *casi seguro* de algunos es el
   *puede ser* de otros"). Los LLMs suelen colapsarla.
4. **Ancho del rango** (`max - min`) — en humanos varía sistemáticamente por expresión.
5. **Coherencia interna** — proporción de casos con `min ≤ puntual ≤ max` para la misma persona y la
   misma palabra. En el estudio original fue **81%**. Un modelo con 100% está siendo *más* coherente
   que los humanos, y eso también es un hallazgo.

In [ ]:
def masa_midpoint(s: pd.Series, lo: int = 45, hi: int = 55) -> float:
    return s.between(lo, hi).mean() * 100


def exacto_50(s: pd.Series) -> float:
    return (s == 50).mean() * 100


sim_punto = simulados[simulados["valid"] & (simulados["bloque"] == "punto")]
hum_punto = humanos.dropna(subset=["punto"])


def resumen_punto(df: pd.DataFrame, etiqueta: str) -> pd.DataFrame:
    g = df.groupby("palabra")["punto"]
    return pd.DataFrame({
        (etiqueta, "n")       : g.size(),
        (etiqueta, "media")   : g.mean().round(1),
        (etiqueta, "sd")      : g.std().round(1),
        (etiqueta, "% 45-55") : g.apply(masa_midpoint).round(1),
        (etiqueta, "% = 50")  : g.apply(exacto_50).round(1),
    })


orden = (hum_punto.groupby("palabra")["punto"].mean().sort_values(ascending=False).index.tolist())

tabla = resumen_punto(hum_punto, "Humanos").join(resumen_punto(sim_punto, "Modelo")).loc[orden]
tabla.columns = pd.MultiIndex.from_tuples(tabla.columns)
tabla

In [ ]:
# Resumen global: los tres números que resumen el ejercicio
print("                          Humanos   Modelo")
print(f"Masa en [45, 55]         {masa_midpoint(hum_punto['punto']):7.1f}% {masa_midpoint(sim_punto['punto']):7.1f}%")
print(f"Exactamente 50           {exacto_50(hum_punto['punto']):7.1f}% {exacto_50(sim_punto['punto']):7.1f}%")
print(f"SD promedio (por expr.)  {hum_punto.groupby('palabra')['punto'].std().mean():7.1f}  {sim_punto.groupby('palabra')['punto'].std().mean():7.1f}")
print(f"Valores distintos usados {hum_punto['punto'].nunique():7d}  {sim_punto['punto'].nunique():7d}")

## 9. Figuras

In [ ]:
COLOR_HUM = "#546E7A"   # mismos colores que el ejercicio 1
COLOR_MOD = "#E64A19"


def _mm(df):
    g = df.groupby("palabra")["punto"]
    return g.mean().reindex(orden), g.std().reindex(orden)


# ── Fig 1 — Perfil de las 16 expresiones: media ± 1 SD ──────────────────────
hm, hs = _mm(hum_punto)
mm, ms = _mm(sim_punto)
y = np.arange(len(orden))

fig, ax = plt.subplots(figsize=(9, 8))
ax.axvspan(45, 55, color="#FFC107", alpha=0.15, zorder=0)
ax.axvline(50, color="#FFC107", lw=1.2, ls="--", zorder=1)
ax.errorbar(hm, y - 0.16, xerr=hs, fmt="o", color=COLOR_HUM, capsize=3,
            label="Humanos", ms=6, zorder=3)
ax.errorbar(mm, y + 0.16, xerr=ms, fmt="s", color=COLOR_MOD, capsize=3,
            label=f"Modelo ({MODEL}, {VERSION_PROMPT})", ms=6, zorder=3)
ax.set_yticks(y); ax.set_yticklabels(orden)
ax.set_xlim(-2, 102); ax.set_xlabel("Probabilidad asignada (0–100)")
ax.invert_yaxis()
ax.set_title("Perfil de las expresiones — media ± 1 desvío estándar\n"
             "La banda amarilla es la zona de punto medio [45, 55]",
             fontsize=12, fontweight="bold")
ax.legend(loc="lower right"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("fig1_perfil_expresiones.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Fig 2 — Dispersión entre participantes ──────────────────────────────────
# El hallazgo central del estudio original. Barras más cortas en el modelo =
# aplanamiento de la variabilidad intragrupo.
fig, ax = plt.subplots(figsize=(10, 5))
w = 0.38
x = np.arange(len(orden))
ax.bar(x - w/2, hs.values, width=w, color=COLOR_HUM, alpha=0.85, label="Humanos")
ax.bar(x + w/2, ms.values, width=w, color=COLOR_MOD, alpha=0.85,
       label=f"Modelo ({VERSION_PROMPT})")
ax.set_xticks(x); ax.set_xticklabels(orden, rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Desvío estándar entre participantes")
ax.set_title("¿Reproduce el modelo el desacuerdo entre personas?",
             fontsize=12, fontweight="bold")
ax.legend(); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("fig2_dispersion.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Fig 3 — Las dos métricas de punto medio ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, fn, titulo in [
    (axes[0], masa_midpoint, "Masa en la zona de punto medio [45, 55]"),
    (axes[1], exacto_50,     "Respuestas exactamente iguales a 50"),
]:
    h = hum_punto.groupby("palabra")["punto"].apply(fn).reindex(orden)
    m = sim_punto.groupby("palabra")["punto"].apply(fn).reindex(orden)
    yy = np.arange(len(orden))
    ax.barh(yy - 0.19, h.values, height=0.38, color=COLOR_HUM, alpha=0.85, label="Humanos")
    ax.barh(yy + 0.19, m.values, height=0.38, color=COLOR_MOD, alpha=0.85,
            label=f"Modelo ({VERSION_PROMPT})")
    ax.set_yticks(yy); ax.set_yticklabels(orden, fontsize=9)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title(titulo, fontsize=11, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend()
fig.suptitle("Sesgo de punto medio por expresión", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig3_punto_medio.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Fig 4 — Distribuciones crudas ───────────────────────────────────────────
# Cada punto es una respuesta. Acá se ve si el modelo está apilando respuestas
# sobre unos pocos valores redondos (50, 70, 90) mientras los humanos se desparraman.
comp = pd.concat([
    hum_punto.assign(Fuente="Humanos")[["palabra", "punto", "Fuente"]],
    sim_punto.assign(Fuente="Modelo")[["palabra", "punto", "Fuente"]],
])

fig, ax = plt.subplots(figsize=(10, 8))
ax.axvspan(45, 55, color="#FFC107", alpha=0.15, zorder=0)
sns.stripplot(data=comp, x="punto", y="palabra", hue="Fuente", order=orden,
              dodge=True, jitter=0.25, size=3.5, alpha=0.6,
              palette=[COLOR_HUM, COLOR_MOD], ax=ax)
ax.set_xlim(-2, 102)
ax.set_xlabel("Probabilidad asignada (0–100)"); ax.set_ylabel("")
ax.set_title("Todas las respuestas, una por punto", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("fig4_distribuciones.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Bloque de rango: ancho y coherencia interna

Las dos métricas de incertidumbre de segundo orden. La coherencia interna es la más interesante:
en el estudio original solo el **81%** de las personas fue coherente consigo misma
(`mín ≤ puntual ≤ máx`). Un modelo que dé 100% no está siendo mejor — está siendo *distinto*.

In [ ]:
sim_rango = simulados[simulados["valid"] & (simulados["bloque"] == "rango")].copy()
hum_rango = humanos.dropna(subset=["min", "max"]).copy()
sim_rango["ancho"] = sim_rango["max"] - sim_rango["min"]
hum_rango["ancho"] = hum_rango["max"] - hum_rango["min"]

orden_r = [p for p in orden if p in set(hum_rango["palabra"])]

# ── Coherencia interna: min <= punto <= max, por persona y expresión ────────
def coherencia(df_punto, df_rango, key_punto="punto"):
    r = df_rango[["participant_id", "palabra", "min", "max"]]
    p = df_punto[["participant_id", "palabra", key_punto]]
    j = r.merge(p, on=["participant_id", "palabra"], how="inner").dropna()
    if not len(j):
        return np.nan, 0
    ok = (j["min"] <= j[key_punto]) & (j[key_punto] <= j["max"])
    return ok.mean() * 100, len(j)


coh_h, n_h = coherencia(hum_punto, hum_rango)
coh_m, n_m = coherencia(sim_punto, sim_rango)

print("                        Humanos   Modelo   (original: 81%)")
print(f"Coherencia interna     {coh_h:7.1f}% {coh_m:7.1f}%   (n={n_h} / {n_m})")
print(f"Ancho medio del rango  {hum_rango['ancho'].mean():7.1f}  {sim_rango['ancho'].mean():7.1f}")

# ── Fig 5 — Ancho del rango por expresión ───────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
w = 0.38
x = np.arange(len(orden_r))
ha = hum_rango.groupby("palabra")["ancho"].mean().reindex(orden_r)
ma = sim_rango.groupby("palabra")["ancho"].mean().reindex(orden_r)
ax.bar(x - w/2, ha.values, width=w, color=COLOR_HUM, alpha=0.85, label="Humanos")
ax.bar(x + w/2, ma.values, width=w, color=COLOR_MOD, alpha=0.85,
       label=f"Modelo ({VERSION_PROMPT})")
ax.set_xticks(x); ax.set_xticklabels(orden_r, rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Ancho medio del rango (máx − mín)")
ax.set_title("¿Varía el ancho del rango según la expresión?",
             fontsize=12, fontweight="bold")
ax.legend(); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("fig5_ancho_rango.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Comparar modelos y versiones de prompt

Cada corrida deja su propio CSV, con `model_name` y `version_prompt` adentro. Para comparar varias,
volvé arriba, cambiá `MODEL` o `VERSION_PROMPT`, corré de nuevo, y después apilá los CSV acá.

Es el mismo patrón que el ejercicio 1: la columna que identifica la corrida es la que permite
apilar todo en un solo dataframe sin reestructurar nada.

In [ ]:
import glob

archivos = sorted(glob.glob("/content/*_expresiones_silicon.csv"))
print("Corridas encontradas:")
for a in archivos:
    print("  ", a)

if len(archivos) > 1:
    todas = pd.concat([pd.read_csv(a) for a in archivos], ignore_index=True)
    todas = todas[todas["valid"] & (todas["bloque"] == "punto")]

    comparacion = todas.groupby(["model_name", "version_prompt"])["punto"].agg(
        n="size",
        media="mean",
        sd_global="std",
    ).round(1)
    comparacion["% 45-55"] = todas.groupby(["model_name", "version_prompt"])["punto"].apply(masa_midpoint).round(1)
    comparacion["% = 50"]  = todas.groupby(["model_name", "version_prompt"])["punto"].apply(exacto_50).round(1)
    comparacion["sd_media_por_expr"] = (
        todas.groupby(["model_name", "version_prompt", "palabra"])["punto"].std()
             .groupby(level=[0, 1]).mean().round(1)
    )

    # Fila de referencia humana
    ref = pd.DataFrame({
        "n": [len(hum_punto)],
        "media": [hum_punto["punto"].mean().round(1)],
        "sd_global": [hum_punto["punto"].std().round(1)],
        "% 45-55": [round(masa_midpoint(hum_punto["punto"]), 1)],
        "% = 50": [round(exacto_50(hum_punto["punto"]), 1)],
        "sd_media_por_expr": [round(hum_punto.groupby("palabra")["punto"].std().mean(), 1)],
    }, index=pd.MultiIndex.from_tuples([("HUMANOS", "—")],
                                       names=["model_name", "version_prompt"]))

    display(pd.concat([ref, comparacion]))
else:
    print("\nHay una sola corrida. Cambiá MODEL o VERSION_PROMPT arriba y volvé a correr "
          "el notebook para poder comparar.")

In [ ]:
l

## 12. Para discutir en clase

1. **¿Dónde está el 50?** Miren la columna `% = 50` de la tabla de la sección 8. ¿En qué expresiones
   lo pone el modelo y en cuáles lo ponen las personas? En el instrumento humano llegar exactamente
   a 50 requiere mover el slider a propósito, porque el círculo arranca oculto: no hay valor por
   defecto que arrastre.

2. **La dispersión.** La figura 2 compara el desacuerdo entre personas contra el desacuerdo entre
   perfiles simulados. Si las barras naranjas son sistemáticamente más cortas, el modelo está
   aplanando la variabilidad intragrupo — el mismo fenómeno que aparece en el ejercicio 1 con la
   WVS. ¿Por qué pasaría eso, si cada llamada recibe un perfil distinto?

3. **¿Sirvió el prompt v2?** Corran las dos versiones y comparen en la sección 11. Ojo con el
   criterio: que la masa en `[45, 55]` baje no alcanza. Si al mismo tiempo el perfil de la figura 1
   se aleja del humano, el prompt no mitigó el sesgo — lo reemplazó por otro.

4. **El perfil es pobre.** El formulario releva edad, género, educación y país. La WVS del ejercicio 1
   además tenía ocupación, clase e ideología. ¿Cuánto de la variabilidad humana es razonable esperar
   que reproduzca un modelo con estas cuatro variables? ¿Qué agregarían al formulario?

5. **Coherencia interna.** El 81% del estudio original significa que 1 de cada 5 personas se
   contradijo entre su valor puntual y su rango. Si el modelo da 100%, ¿es mejor o simplemente no
   está simulando personas?

6. **Y finalmente:** la muestra humana son ustedes. n chico, y nada parecido a una muestra
   probabilística de nada. ¿Qué conclusiones aguanta este diseño y cuáles no?